# Ch 12 — Consumption Smoothing (Friedman–Hall)

A hands-on companion to the QuantEcon lecture. The idea: a household with a fixed
borrowing/lending rate and a known income path chooses a *constant* consumption level
so that the present value of consumption equals financial wealth plus the present
value of future income ("human wealth"). Run the cells below, then try changing
the income path or interest rate at the bottom.

This is a from-scratch implementation for practice — not copied from the lecture's code.

In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
except ImportError:
    import piplite
    await piplite.install(['numpy', 'matplotlib'])
    import numpy as np
    import matplotlib.pyplot as plt

In [ ]:
def solve_consumption_path(a0, income_path, R):
    """
    a0           : initial financial wealth (can be negative, e.g. student debt)
    income_path  : array of non-financial income y_0..y_T
    R            : gross interest rate (e.g. 1.05 for 5%)

    Returns (c_const, wealth_path, human_wealth).
    """
    T = len(income_path) - 1
    disc = np.array([R ** (-t) for t in range(T + 1)])

    human_wealth = disc @ income_path
    annuity_factor = (1 - 1 / R) / (1 - (1 / R) ** (T + 1))
    c_const = annuity_factor * (a0 + human_wealth)

    wealth_path = np.empty(T + 2)
    wealth_path[0] = a0
    for t in range(T + 1):
        wealth_path[t + 1] = R * (wealth_path[t] + income_path[t] - c_const)

    return c_const, wealth_path, human_wealth

In [ ]:
def plot_consumption_path(a0, income_path, R):
    T = len(income_path) - 1
    c_const, wealth_path, human_wealth = solve_consumption_path(a0, income_path, R)

    print(f"constant optimal consumption c* = {c_const:.4f}")
    print(f"human wealth (PV of income)   = {human_wealth:.4f}")
    print(f"terminal financial wealth     = {wealth_path[-1]:.6f}  (should be ~0)")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    axes[0].plot(range(T + 1), income_path, label="income $y_t$", lw=2)
    axes[0].axhline(c_const, color="tab:orange", lw=2, label="consumption $c_t$")
    axes[0].set_xlabel("t"); axes[0].legend(); axes[0].set_title("Income vs. consumption")

    axes[1].plot(range(T + 2), wealth_path, color="green", lw=2, label="financial wealth $a_t$")
    axes[1].axhline(0, color="black", lw=1, ls="--")
    axes[1].set_xlabel("t"); axes[1].legend(); axes[1].set_title("Financial wealth path")

    plt.tight_layout()
    plt.show()

## Try it: a working life with retirement

Starts $2 in debt, earns 1 unit/year for 46 years, then retires (income drops to 0).

In [ ]:
a0 = -2.0
R = 1.05
income_path = np.concatenate([np.ones(46), np.zeros(20)])

plot_consumption_path(a0, income_path, R)

## Now experiment

Edit the cell below — try a one-time windfall, a permanent raise, or a higher interest
rate — and re-run to see how the flat consumption line shifts.

In [ ]:
# Example: a one-time windfall of +2.5 in year 21
custom_income = np.concatenate([np.ones(21), np.array([2.5]), np.ones(24), np.zeros(20)])
plot_consumption_path(a0=-2.0, income_path=custom_income, R=1.05)